## Decisions

Should I users/machines/others or combinaitons of these. This notebook will provide the reserach and evidence for which one to choose.

In [1]:
# Packages 
import polars as pl
import matplotlib.pyplot as plt

In [23]:
# Load data
df = pl.scan_csv("../auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [24]:
# First 5 rows of dataset
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
1,"""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
1,"""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
1,"""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [14]:
# What are the different type of source users
types = df.with_columns(
    pl.col("src_user")
      .str.strip_chars('"')
      .str.split("@").list.first()
      .alias("src_identifier")
).with_columns(
    pl.when(pl.col("src_identifier").str.ends_with("$"))
      .then(pl.lit("machine"))
      .when(pl.col("src_identifier").str.starts_with("U"))
      .then(pl.lit("user"))
      .otherwise(pl.lit("other"))
      .alias("src_type")
)

types.filter(pl.col("src_type") == "other") \
  .select("src_identifier") \
  .unique() \
  .collect(engine="streaming")

src_identifier
str
"""LOCAL SERVICE"""
"""NETWORK SERVICE"""
"""SYSTEM"""
"""ANONYMOUS LOGON"""


The distinct source users are:

USERS, MACHINES, SYSTEM, LOCAL SERVICE, ANONYMOUS LOGON, NETWORK SERVICE

In [15]:
types.group_by("src_type").len().sort("len", descending=True).collect(engine="streaming")

src_type,len
str,u32
"""machine""",633193503
"""user""",341692445
"""other""",76544511


633193503 (~600 million) machine users.

341692445 (~ 300 million) human users.

76544511 (~ 76 million) others

In [17]:
types.filter(pl.col("src_type") == "other") \
  .group_by("src_identifier").len() \
  .sort("len", descending=True) \
  .collect(engine="streaming")

src_identifier,len
str,u32
"""NETWORK SERVICE""",39754826
"""ANONYMOUS LOGON""",33534047
"""LOCAL SERVICE""",2870066
"""SYSTEM""",385572


39754826 (~39 million) NETWORK SERVICE	

33534047 (~ 33 million) ANONYMOUS LOGON

2870066 (~ 280k) LOCAL SERVICE

385572 (~ 38k) SYSTEM

Failure rates are so low, that looking at failure rates across users/machines/others doesn't yield anything useful.